In [2]:
import pandas as pd
from requests.auth import HTTPBasicAuth
import requests as re

In [3]:
df = pd.read_csv(r'df_final_com_cnpj.csv')
display(df)

,Unnamed: 0,ROE,ROA,Margem Liquida,Margem Bruta,Divida Liquida,EBITDA,Divida_Liquida_EBITDA,Liquidez Corrente,PL/Ativos,...,Dividendos_Pagos_Total,Total_Acoes,VPA,LPA,Giro Ativos,Ticker_CVM,Ano,Dividendos_Pagos_T1,CNPJ,ticker
0,0,0.000,0.000,0.000,27.498,-14171200.0,35524338.0,-0.399,74.806,84.045,...,8997310.0,0.000000e+00,0.000,0.000,8.477,1023,2019,6070660.0,191,BBAS3
1,1,78.681,0.785,13.473,56.181,585199400.0,-4678300.0,-125.088,56.866,0.997,...,6070660.0,2.853242e+09,0.006,0.005,5.825,1023,2020,7124615.0,191,BBAS3
2,2,101.201,1.038,15.660,47.380,653247000.0,4036400.0,161.839,26.542,1.026,...,7124615.0,2.853442e+09,0.007,0.007,6.631,1023,2021,13175440.0,191,BBAS3
3,3,135.941,1.487,12.619,31.430,734952500.0,17571900.0,41.825,24.894,1.094,...,13175440.0,2.853636e+09,0.008,0.010,11.782,1023,2022,15358300.0,191,BBAS3
4,4,152.227,1.540,12.495,33.435,794616300.0,17520500.0,45.354,21.915,1.012,...,15358300.0,2.853776e+09,0.008,0.012,12.324,1023,2023,16563560.0,191,BBAS3
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4118,4118,8.212,-9.171,-19.269,3.199,-53.0,-427773.0,0.000,0.490,-111.687,...,0.0,0.000000e+00,0.000,0.000,47.597,9989,2019,0.0,33412081000196,RPMG3
4119,4119,12.928,-12.761,-27.609,-7.380,-9.0,-660901.0,0.000,0.516,-98.704,...,0.0,6.768500e+04,-47.187,-6.100,46.220,9989,2020,0.0,33412081000196,RPMG3
4120,4120,10.174,-8.620,-16.360,-2.776,-13.0,-610057.0,0.000,0.553,-84.725,...,0.0,6.768500e+04,-52.533,-5.345,52.687,9989,2021,0.0,33412081000196,RPMG3
4121,4121,20.835,-18.551,-18.786,-12.681,1895.0,-1305920.0,-0.001,0.525,-89.036,...,0.0,6.768500e+04,-66.361,-13.827,98.747,9989,2022,0.0,33412081000196,RPMG3


In [4]:
import pandas as pd
import yfinance as yf

df['Ano'] = df['Ano'].astype(int)
df['ticker'] = df['ticker'].astype(str).str.strip()

tickers_unicos = df['ticker'].dropna().unique()
mapa_precos = {}

for ticker in tickers_unicos:
    try:
        hist = yf.download(f"{ticker}.SA", start="2010-01-01", progress=False)
        
        if hist.empty:
            continue

        if isinstance(hist.columns, pd.MultiIndex):
            s_preco = hist['Adj Close'].iloc[:, 0] if 'Adj Close' in hist.columns.levels[0] else hist['Close'].iloc[:, 0]
        else:
            s_preco = hist['Adj Close'] if 'Adj Close' in hist.columns else hist['Close']

        s_preco_anual = s_preco.groupby(s_preco.index.year).tail(1)
        
        for data, preco in s_preco_anual.items():
            mapa_precos[(ticker, data.year)] = float(preco)
            
    except Exception as e:
        print(f"Erro ao processar {ticker}: {e}")

df['Preco_Fechamento'] = df.apply(lambda x: mapa_precos.get((x['ticker'], x['Ano'])), axis=1)
df_final = df.dropna(subset=['Preco_Fechamento']).reset_index(drop=True)

print(f"Preços mapeados: {len(mapa_precos)}")

/Users/fratucci/Projetos/PUC-Minas/.venv/lib/python3.14/site-packages/yfinance/scrapers/history.py:144: Pandas4Warning: Timestamp.utcnow is deprecated and will be removed in a future version. Use Timestamp.now('UTC') instead.
  end_dt = pd.Timestamp.utcnow().tz_convert(tz)
/Users/fratucci/Projetos/PUC-Minas/.venv/lib/python3.14/site-packages/yfinance/scrapers/history.py:201: Pandas4Warning: Timestamp.utcnow is deprecated and will be removed in a future version. Use Timestamp.now('UTC') instead.
  dt_now = pd.Timestamp.utcnow()
/Users/fratucci/Projetos/PUC-Minas/.venv/lib/python3.14/site-packages/yfinance/scrapers/history.py:144: Pandas4Warning: Timestamp.utcnow is deprecated and will be removed in a future version. Use Timestamp.now('UTC') instead.
  end_dt = pd.Timestamp.utcnow().tz_convert(tz)
/Users/fratucci/Projetos/PUC-Minas/.venv/lib/python3.14/site-packages/yfinance/scrapers/history.py:201: Pandas4Warning: Timestamp.utcnow is deprecated and will be removed in a future version. U

Preços mapeados: 3535


In [5]:
df['Valor_de_Mercado'] = df['Preco_Fechamento'] * df['Total_Acoes']
df['Dividend_Yield_T1'] = df['Dividendos_Pagos_T1'] / df['Valor_de_Mercado']
df['Y_Bom_Pagador'] = (df['Dividend_Yield_T1'] >= 0.06).astype(int)

df_modelo_logistico = df.dropna(subset=['Y_Bom_Pagador', 'Dividend_Yield_T1']).copy()

display(df_modelo_logistico)

,Unnamed: 0,ROE,ROA,Margem Liquida,Margem Bruta,Divida Liquida,EBITDA,Divida_Liquida_EBITDA,Liquidez Corrente,PL/Ativos,...,Giro Ativos,Ticker_CVM,Ano,Dividendos_Pagos_T1,CNPJ,ticker,Preco_Fechamento,Valor_de_Mercado,Dividend_Yield_T1,Y_Bom_Pagador
0,0,0.000,0.000,0.000,27.498,-14171200.0,35524338.0,-0.399,74.806,84.045,...,8.477,1023,2019,6070660.0,191,BBAS3,16.824217,0.000000e+00,inf,1
1,1,78.681,0.785,13.473,56.181,585199400.0,-4678300.0,-125.088,56.866,0.997,...,5.825,1023,2020,7124615.0,191,BBAS3,12.856666,3.668317e+10,0.000194,0
2,2,101.201,1.038,15.660,47.380,653247000.0,4036400.0,161.839,26.542,1.026,...,6.631,1023,2021,13175440.0,191,BBAS3,10.299092,2.938787e+10,0.000448,0
3,3,135.941,1.487,12.619,31.430,734952500.0,17571900.0,41.825,24.894,1.094,...,11.782,1023,2022,15358300.0,191,BBAS3,13.899334,3.966364e+10,0.000387,0
4,4,152.227,1.540,12.495,33.435,794616300.0,17520500.0,45.354,21.915,1.012,...,12.324,1023,2023,16563560.0,191,BBAS3,24.114315,6.881685e+10,0.000241,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4110,4110,15.951,8.657,8.156,17.000,102777200.0,86911900.0,1.183,1.681,54.274,...,106.143,9539,2023,42454800.0,88613658000110,PTNT3,10.351127,4.973353e+08,0.085365,1
4119,4119,12.928,-12.761,-27.609,-7.380,-9.0,-660901.0,0.000,0.516,-98.704,...,46.220,9989,2020,0.0,33412081000196,RPMG3,2.820000,1.908717e+05,0.000000,0
4120,4120,10.174,-8.620,-16.360,-2.776,-13.0,-610057.0,0.000,0.553,-84.725,...,52.687,9989,2021,0.0,33412081000196,RPMG3,2.800000,1.895180e+05,0.000000,0
4121,4121,20.835,-18.551,-18.786,-12.681,1895.0,-1305920.0,-0.001,0.525,-89.036,...,98.747,9989,2022,0.0,33412081000196,RPMG3,1.700000,1.150645e+05,0.000000,0


In [13]:
pd.set_option('display.max_columns', None)
pd.set_option('display.expand_frame_repr', False)
df_modelo_logistico.describe()

/Users/fratucci/Projetos/PUC-Minas/.venv/lib/python3.14/site-packages/numpy/lib/_function_base_impl.py:4596: RuntimeWarning: invalid value encountered in subtract
  diff_b_a = b - a
/Users/fratucci/Projetos/PUC-Minas/.venv/lib/python3.14/site-packages/numpy/_core/_methods.py:49: RuntimeWarning: invalid value encountered in reduce
  return umr_sum(a, axis, dtype, out, keepdims, initial, where)


,Unnamed: 0,ROE,ROA,Margem Liquida,Margem Bruta,Divida Liquida,EBITDA,Divida_Liquida_EBITDA,Liquidez Corrente,PL/Ativos,Passivos/Ativos,Margem Ebit,Lucro Líquido,Dividendos Propostos (BPP),Dividendos a Pagar (BPP),Dividendos Pagos (DFC),Payout Contábil (%),Payout Caixa (%),FCL,Dividendos_Pagos_Total,Total_Acoes,VPA,LPA,Giro Ativos,Ticker_CVM,Ano,Dividendos_Pagos_T1,CNPJ,Preco_Fechamento,Valor_de_Mercado,Dividend_Yield_T1,Y_Bom_Pagador
count,1778.000000,1778.000000,1778.000000,1778.000000,1778.000000,1.778000e+03,1.778000e+03,1778.000000,1778.000000,1778.000000,1778.0,1778.000000,1.778000e+03,1.778000e+03,1.778000e+03,1.778000e+03,1778.000000,1778.000000,1.778000e+03,1.778000e+03,1.778000e+03,1778.000000,1778.000000,1778.000000,1778.000000,1778.000000,1.778000e+03,1.778000e+03,1.778000e+03,1.778000e+03,1778.000000,1778.000000
mean,1989.424634,-54.367599,3.868304,2.319976,29.787726,7.681249e+06,2.754227e+06,16.836024,2.471435,35.066929,100.0,-19.714119,2.614271e+06,9.623707e+04,5.186103e+05,1.435243e+05,17.786023,173.109428,2.123873e+06,1.102933e+06,2.455646e+08,-5.402443,-0.118588,63.655057,15819.768841,2018.667042,1.279612e+06,4.063883e+13,7.428322e+03,1.213413e+10,NaN,0.525872
std,1290.714281,2648.915729,12.605433,391.288908,116.354067,4.842624e+07,1.462734e+07,351.425216,4.527051,48.823812,0.0,289.188052,1.784372e+07,7.254190e+05,4.360051e+06,1.026328e+07,131.816127,8032.877296,1.547106e+07,6.746382e+06,9.487130e+08,150.052984,11.456794,52.715834,8069.467279,3.478602,7.546162e+06,3.181210e+13,2.472128e+05,1.392230e+11,NaN,0.499471
min,0.000000,-111510.278000,-125.941000,-8692.164000,-3678.628000,-1.299782e+08,-7.545090e+07,-805.827000,0.011000,-495.454000,100.0,-5894.607000,-4.599660e+07,-8.257000e+03,0.000000e+00,-7.328700e+07,-1063.915000,-143488.682000,-1.213010e+08,0.000000e+00,0.000000e+00,-3812.883000,-317.475000,-5.552000,94.000000,2012.000000,0.000000e+00,1.910000e+02,-2.309711e+05,-1.346562e+12,-inf,0.000000
25%,864.250000,3.548500,0.929000,1.443250,21.665250,5.540775e+04,1.595200e+04,-0.240000,1.219500,28.159250,100.0,-28.266250,1.322675e+04,0.000000e+00,0.000000e+00,-7.986900e+04,0.000000,0.000000,-6.377750e+04,4.060000e+02,0.000000e+00,0.000000,0.000000,31.347750,8753.000000,2016.000000,2.619250e+03,8.801621e+12,5.181318e+00,0.000000e+00,0.000015,0.000000
50%,1801.500000,11.169000,4.260500,7.532500,30.749500,6.521450e+05,2.097565e+05,1.619500,1.756500,41.971500,100.0,-15.605000,1.503065e+05,0.000000e+00,8.500000e+01,0.000000e+00,1.434000,0.390500,5.086950e+04,4.163150e+04,7.100000e+03,0.000000,0.000000,53.004000,18660.000000,2020.000000,5.016450e+04,3.360201e+13,1.008041e+01,7.139376e+04,0.114480,1.000000
75%,3131.750000,19.387000,8.253750,15.670500,47.109750,3.070704e+06,9.744095e+05,4.726500,2.501250,56.739250,100.0,-8.069750,6.347838e+05,0.000000e+00,3.727400e+04,0.000000e+00,23.938750,36.227750,4.544792e+05,2.692032e+05,7.361828e+07,0.013000,0.002000,83.962500,21334.000000,2022.000000,3.027750e+05,6.148665e+13,1.875932e+01,5.653597e+08,NaN,1.000000
max,4122.000000,3580.955000,167.023000,7364.260000,100.000000,7.946163e+08,2.878960e+08,11043.982000,99.816000,94.839000,100.0,5774.862000,3.692450e+08,1.784900e+07,8.770090e+07,3.081130e+08,4974.954000,300802.439000,2.510330e+08,1.946090e+08,1.304420e+10,198.309000,135.273000,450.431000,80187.000000,2023.000000,1.946090e+08,9.783718e+13,9.845848e+06,4.127014e+12,inf,1.000000


In [14]:
df_modelo_logistico.shape

(1778, 33)

In [19]:
import numpy as np

df_modelo_logistico = df_modelo_logistico[(df_modelo_logistico['Total_Acoes'] != 0) & (df_modelo_logistico['Dividend_Yield_T1'] != -np.inf)]

In [20]:
df_modelo_logistico.shape

(926, 33)

In [21]:
import statsmodels.formula.api as smf
import numpy as np
import pandas as pd

df_reg = df_modelo_logistico.copy()
df_reg.columns = df_reg.columns.str.replace(' ', '_').str.replace('/', '_')

df_reg = df_reg.replace([np.inf, -np.inf], np.nan).dropna(
    subset=['Y_Bom_Pagador', 'ROE', 'Margem_Liquida', 'Divida_Liquida_EBITDA', 'Liquidez_Corrente']
)

formula = 'Y_Bom_Pagador ~ ROE + Margem_Liquida + Divida_Liquida_EBITDA + Liquidez_Corrente'

modelo = smf.logit(formula=formula, data=df_reg).fit()

print(modelo.summary())

odds_ratios = pd.DataFrame({
    "Odds_Ratio": np.exp(modelo.params),
    "P_Value": round(modelo.pvalues, 4)
})
display(odds_ratios)

Optimization terminated successfully.
         Current function value: 0.319894
         Iterations 10
                           Logit Regression Results                           
Dep. Variable:          Y_Bom_Pagador   No. Observations:                  926
Model:                          Logit   Df Residuals:                      921
Method:                           MLE   Df Model:                            4
Date:                Sat, 21 Mar 2026   Pseudo R-squ.:                0.004283
Time:                        10:43:57   Log-Likelihood:                -296.22
converged:                       True   LL-Null:                       -297.50
Covariance Type:            nonrobust   LLR p-value:                    0.6360
                            coef    std err          z      P>|z|      [0.025      0.975]
-----------------------------------------------------------------------------------------
Intercept                -2.1188      0.150    -14.151      0.000      -2.412      -1

,Odds_Ratio,P_Value
Intercept,0.120171,0.0000
ROE,1.000273,0.6039
Margem_Liquida,1.000122,0.5534
Divida_Liquida_EBITDA,0.999324,0.7384
Liquidez_Corrente,0.957332,0.3716


In [23]:
import statsmodels.formula.api as smf
import numpy as np
import pandas as pd

df_reg = df_modelo_logistico.copy()

mapeamento_colunas = {
    'Margem Liquida': 'Margem_Liquida',
    'Payout Contábil (%)': 'Payout_Contabil',
    'Giro Ativos': 'Giro_Ativos'
}
df_reg = df_reg.rename(columns=mapeamento_colunas)

df_reg = df_reg.replace([np.inf, -np.inf], np.nan).dropna(
    subset=['Y_Bom_Pagador', 'ROE', 'Margem_Liquida', 'Divida_Liquida_EBITDA', 'Payout_Contabil', 'FCL', 'LPA']
)

formula = 'Y_Bom_Pagador ~ ROE + Margem_Liquida + Divida_Liquida_EBITDA + Payout_Contabil + FCL + LPA'

modelo_novo = smf.logit(formula=formula, data=df_reg).fit()

print(modelo_novo.summary())

odds_ratios = pd.DataFrame({
    "Odds_Ratio": np.exp(modelo_novo.params),
    "P_Value": round(modelo_novo.pvalues, 4)
})
display(odds_ratios)

Optimization terminated successfully.
         Current function value: 0.307945
         Iterations 9
                           Logit Regression Results                           
Dep. Variable:          Y_Bom_Pagador   No. Observations:                  926
Model:                          Logit   Df Residuals:                      919
Method:                           MLE   Df Model:                            6
Date:                Sat, 21 Mar 2026   Pseudo R-squ.:                 0.04148
Time:                        10:51:14   Log-Likelihood:                -285.16
converged:                       True   LL-Null:                       -297.50
Covariance Type:            nonrobust   LLR p-value:                 0.0003917
                            coef    std err          z      P>|z|      [0.025      0.975]
-----------------------------------------------------------------------------------------
Intercept                -2.3448      0.120    -19.582      0.000      -2.580      -2.

,Odds_Ratio,P_Value
Intercept,0.095866,0.0000
ROE,1.000268,0.6302
Margem_Liquida,1.000040,0.8557
Divida_Liquida_EBITDA,0.999536,0.7280
Payout_Contabil,1.000418,0.2927
FCL,1.000000,0.0048
LPA,1.059117,0.0026


In [24]:
import statsmodels.formula.api as smf
import numpy as np
import pandas as pd

df_reg = df_modelo_logistico.copy()

mapeamento_colunas = {
    'Margem Liquida': 'Margem_Liquida',
    'Payout Contábil (%)': 'Payout_Contabil',
    'Giro Ativos': 'Giro_Ativos'
}
df_reg = df_reg.rename(columns=mapeamento_colunas)

cols = ['ROE', 'Margem_Liquida', 'Divida_Liquida_EBITDA', 'Payout_Contabil', 'FCL', 'LPA']

df_reg = df_reg.replace([np.inf, -np.inf], np.nan).dropna(subset=['Y_Bom_Pagador'] + cols)

for col in cols:
    limite_inferior = df_reg[col].quantile(0.05)
    limite_superior = df_reg[col].quantile(0.95)
    df_reg[col] = df_reg[col].clip(lower=limite_inferior, upper=limite_superior)

formula = 'Y_Bom_Pagador ~ ROE + Margem_Liquida + Divida_Liquida_EBITDA + Payout_Contabil + FCL + LPA'

modelo_winsorizado = smf.logit(formula=formula, data=df_reg).fit()

print(modelo_winsorizado.summary())

odds_ratios = pd.DataFrame({
    "Odds_Ratio": np.exp(modelo_winsorizado.params),
    "P_Value": round(modelo_winsorizado.pvalues, 4)
})
display(odds_ratios)

Optimization terminated successfully.
         Current function value: 0.226993
         Iterations 8
                           Logit Regression Results                           
Dep. Variable:          Y_Bom_Pagador   No. Observations:                  926
Model:                          Logit   Df Residuals:                      919
Method:                           MLE   Df Model:                            6
Date:                Sat, 21 Mar 2026   Pseudo R-squ.:                  0.2935
Time:                        10:52:20   Log-Likelihood:                -210.20
converged:                       True   LL-Null:                       -297.50
Covariance Type:            nonrobust   LLR p-value:                 4.752e-35
                            coef    std err          z      P>|z|      [0.025      0.975]
-----------------------------------------------------------------------------------------
Intercept                -3.8652      0.277    -13.970      0.000      -4.407      -3.

,Odds_Ratio,P_Value
Intercept,0.020959,0.0000
ROE,1.005866,0.3474
Margem_Liquida,1.004358,0.5463
Divida_Liquida_EBITDA,1.009868,0.6791
Payout_Contabil,1.028210,0.0001
FCL,1.000000,0.0003
LPA,2.093521,0.0000


In [25]:
import statsmodels.formula.api as smf
import numpy as np
import pandas as pd

df_reg = df_modelo_logistico.copy()

mapeamento_colunas = {
    'Payout Contábil (%)': 'Payout_Contabil'
}
df_reg = df_reg.rename(columns=mapeamento_colunas)

cols = ['Payout_Contabil', 'FCL', 'LPA']

df_reg = df_reg.replace([np.inf, -np.inf], np.nan).dropna(subset=['Y_Bom_Pagador'] + cols)

for col in cols:
    limite_inferior = df_reg[col].quantile(0.05)
    limite_superior = df_reg[col].quantile(0.95)
    df_reg[col] = df_reg[col].clip(lower=limite_inferior, upper=limite_superior)

formula = 'Y_Bom_Pagador ~ Payout_Contabil + FCL + LPA'

modelo_final = smf.logit(formula=formula, data=df_reg).fit()

print(modelo_final.summary())

odds_ratios = pd.DataFrame({
    "Odds_Ratio": np.exp(modelo_final.params),
    "P_Value": round(modelo_final.pvalues, 4)
})
display(odds_ratios)

Optimization terminated successfully.
         Current function value: 0.227803
         Iterations 7
                           Logit Regression Results                           
Dep. Variable:          Y_Bom_Pagador   No. Observations:                  926
Model:                          Logit   Df Residuals:                      922
Method:                           MLE   Df Model:                            3
Date:                Sat, 21 Mar 2026   Pseudo R-squ.:                  0.2909
Time:                        10:55:20   Log-Likelihood:                -210.95
converged:                       True   LL-Null:                       -297.50
Covariance Type:            nonrobust   LLR p-value:                 2.724e-37
                      coef    std err          z      P>|z|      [0.025      0.975]
-----------------------------------------------------------------------------------
Intercept          -3.7168      0.234    -15.889      0.000      -4.175      -3.258
Payout_Contabi

,Odds_Ratio,P_Value
Intercept,0.024311,0.0000
Payout_Contabil,1.028801,0.0000
FCL,1.000000,0.0001
LPA,2.106290,0.0000
